这版模型的目标不是精确预测：明天隔夜收益率是多少，而是预测：这只股票在当天横截面里应该排在前面还是后面

输出标签：target_cs_rank

| `target_cs_rank` 数值 | 含义                |
| ------------------: | ----------------- |
|           接近 `+0.5` | 这只股票当天真实隔夜收益率排名靠前 |
|              接近 `0` | 排名中间              |
|           接近 `-0.5` | 这只股票当天真实隔夜收益率排名靠后 |


固定使用的全局参数

| 数据集        | 时间范围                    | 用途       |
| ---------- | ----------------------- | -------- |
| train      | 2010-01-04 到 2018-12-31 | 用来训练模型   |
| validation | 2019-01-02 到 2021-12-31 | 用来选择最优参数 |
| test       | 2022-01-03 到 2024 年底    | 最终样本外检验  |

设置决策树的数量：在100, 200, 400, 600, 800这组参数中逐个试一遍，找到最优


一共训练23 组候选参数组合

然后用 validation set 上的结果选择最优模型。

这 23 组参数主要围绕 7 个核心超参数变化：

n_estimators 决策树数量

max_depth 决策树最大深度

learning_rate 梯度下降步长

subsample

colsample_bytree

min_child_weight

reg_lambda

reg_alpha

gamma

23个参数组合的具体配置如下：


| model_id | n_estimators | max_depth | learning_rate | subsample | colsample_bytree | min_child_weight | reg_lambda | reg_alpha | gamma |
| -------: | -----------: | --------: | ------------: | --------: | ---------------: | ---------------: | ---------: | --------: | ----: |
|        1 |          100 |         2 |          0.05 |      0.90 |             0.90 |                1 |          1 |         0 |     0 |
|        2 |          200 |         2 |          0.03 |      0.90 |             0.90 |                1 |          1 |         0 |     0 |
|        3 |          400 |         2 |          0.02 |      0.90 |             0.90 |                3 |          3 |         0 |     0 |
|        4 |          100 |         3 |          0.05 |      0.90 |             0.90 |                1 |          1 |         0 |     0 |
|        5 |          200 |         3 |          0.03 |      0.90 |             0.90 |                1 |          1 |         0 |     0 |
|        6 |          400 |         3 |          0.02 |      0.90 |             0.90 |                3 |          3 |         0 |     0 |
|        7 |          600 |         3 |          0.01 |      0.90 |             0.90 |                5 |          5 |         0 |     0 |
|        8 |          200 |         3 |          0.03 |      0.80 |             0.80 |                5 |          5 |         0 |     0 |
|        9 |          400 |         3 |          0.02 |      0.80 |             0.80 |                5 |         10 |       0.1 |     0 |
|       10 |          600 |         3 |          0.01 |      0.80 |             0.80 |               10 |         10 |       0.1 |     0 |
|       11 |          100 |         4 |          0.05 |      0.90 |             0.90 |                1 |          1 |         0 |     0 |
|       12 |          200 |         4 |          0.03 |      0.90 |             0.90 |                3 |          3 |         0 |     0 |
|       13 |          400 |         4 |          0.02 |      0.85 |             0.85 |                5 |          5 |         0 |     0 |
|       14 |          600 |         4 |          0.01 |      0.85 |             0.85 |                5 |         10 |       0.1 |     0 |
|       15 |          200 |         4 |          0.03 |      0.80 |             0.80 |               10 |         10 |       0.1 |   0.1 |
|       16 |          400 |         4 |          0.02 |      0.80 |             0.80 |               10 |         20 |       0.1 |   0.1 |
|       17 |          600 |         4 |          0.01 |      0.80 |             0.80 |               20 |         20 |       1.0 |   0.1 |
|       18 |          100 |         5 |          0.05 |      0.85 |             0.85 |                3 |          3 |         0 |     0 |
|       19 |          200 |         5 |          0.03 |      0.85 |             0.85 |                5 |          5 |         0 |     0 |
|       20 |          400 |         5 |          0.02 |      0.80 |             0.80 |               10 |         10 |       0.1 |   0.1 |
|       21 |          800 |         2 |          0.01 |      0.90 |             0.90 |                3 |          3 |         0 |     0 |
|       22 |          800 |         3 |          0.01 |      0.85 |             0.85 |                5 |          5 |         0 |     0 |
|       23 |          800 |         4 |          0.01 |      0.80 |             0.80 |               10 |         10 |       0.1 |   0.1 |

参数选择完成后，代码会取最优参数组合，然后用完整 train set 重新训练最终模型


代码输出的数据文件里面最关键有用的是以下几个：

xgboost2_predictions_test.csv

xgboost2_daily_ic_summary.csv

xgboost2_tuning_results.csv

xgboost2_long_short_summary.csv

xgboost2_metrics.csv

xgboost2_prediction_distribution_summary.csv

xgboost2_feature_importance_gain.csv

In [1]:
# ============================================================
# XGBoost2 Prediction for Overnight Return / Cross-sectional Score
#
# 输入文件：
#   MLF_coursework/stage4_final_75_features_panel.parquet
#
# 输出文件夹：
#   MLF_coursework/stage4_feature_selection_outputs/XGBoost2_prediction_outcome
#
# 核心改进：
#   1. 用 daily cross-sectional rank target 训练，贴合排序任务
#   2. 不使用 RMSE early stopping，避免 best_iteration = 0
#   3. 增加更多参数组合
#   4. 用 validation mean daily Spearman IC 选择最优模型
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pickle
import json

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor


# ============================================================
# 0. 基本设置
# ============================================================

RANDOM_STATE = 42

# 原始收益率目标变量
TARGET_COL = "target_r_on_next_winsor"

# 真实收益率列，用于 long-short 检验
REALIZED_RETURN_COL_CANDIDATE = "target_r_on_next"

# 训练目标模式：
# "cs_rank" 更适合第五步每日横截面排序任务
# "raw_return" 表示直接预测 target_r_on_next_winsor
MODEL_TARGET_MODE = "cs_rank"

# 调参阶段最多使用多少 train rows
# 数据太大时，全量调参会很慢，所以调参阶段先抽样
# 最终模型仍然会用完整 train set 重新训练
TUNING_MAX_TRAIN_ROWS = 600_000

# 调参阶段是否使用完整 validation set
# 建议保持 1.0
TUNING_VALID_FRAC = 1.0

# 最终模型是否用 train + validation 重训
# 为了保持 validation 的纯调参作用，默认 False
# 如果只关心 test 最终表现，可以改成 True
REFIT_FINAL_ON_TRAIN_VALID = False


# ============================================================
# 1. 自动定位项目目录
# ============================================================

def find_project_dir():
    """
    自动寻找 MLF_coursework 项目根目录。
    适用于：
    1. 当前 notebook 在 MLF_coursework 根目录下；
    2. 当前 notebook 在 MLF_coursework 子文件夹下；
    3. 当前目录下可以搜索到 stage4_final_75_features_panel.parquet。
    """
    cwd = Path.cwd().resolve()

    for p in [cwd] + list(cwd.parents):
        if p.name == "MLF_coursework":
            return p

    for p in [cwd] + list(cwd.parents):
        if (p / "stage4_final_75_features_panel.parquet").exists():
            return p

    matches = list(cwd.rglob("stage4_final_75_features_panel.parquet"))
    if len(matches) > 0:
        return matches[0].parent

    raise FileNotFoundError(
        "没有找到 MLF_coursework 项目目录，也没有找到 stage4_final_75_features_panel.parquet。"
    )


PROJECT_DIR = find_project_dir()

DATA_PATH = PROJECT_DIR / "stage4_final_75_features_panel.parquet"

FEATURE_SELECTION_DIR = PROJECT_DIR / "stage4_feature_selection_outputs"
FEATURE_FILE = FEATURE_SELECTION_DIR / "boruta75_feature_cols.txt"

OUTPUT_DIR = FEATURE_SELECTION_DIR / "XGBoost2_prediction_outcome"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Project directory:")
print(PROJECT_DIR)

print("\nInput data path:")
print(DATA_PATH)

print("\nFeature list path:")
print(FEATURE_FILE)

print("\nOutput directory:")
print(OUTPUT_DIR)
print("=" * 100)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"找不到输入数据文件：{DATA_PATH}")


# ============================================================
# 2. 读取数据
# ============================================================

df = pd.read_parquet(DATA_PATH)

print("Original data shape:", df.shape)

if "date" not in df.columns:
    raise ValueError("数据中找不到 date 列。")

df["date"] = pd.to_datetime(df["date"])

sort_cols = ["date"]
if "ticker" in df.columns:
    sort_cols.append("ticker")
elif "instrument_id" in df.columns:
    sort_cols.append("instrument_id")

df = df.sort_values(sort_cols).reset_index(drop=True)

print("Date range:", df["date"].min(), "to", df["date"].max())
print("Number of columns:", len(df.columns))


# ============================================================
# 3. 只保留可交易样本
# ============================================================

if "is_trade_eligible" in df.columns:
    before_rows = len(df)
    df = df[df["is_trade_eligible"].fillna(0).astype(bool)].copy()
    after_rows = len(df)

    print("\nFiltered by is_trade_eligible == True")
    print("Rows before:", before_rows)
    print("Rows after:", after_rows)
else:
    print("\nNo is_trade_eligible column found. Using all rows.")


# ============================================================
# 4. 设置目标变量
# ============================================================

if TARGET_COL not in df.columns:
    raise ValueError(f"找不到目标变量 {TARGET_COL}。")

df = df[df[TARGET_COL].notna()].copy()

# 每日横截面 rank target
# 这一步是本版 XGBoost2 的关键改进
# target_cs_rank 越高，说明当天这只股票的真实隔夜收益率排名越靠前
df["target_cs_rank"] = (
    df.groupby("date")[TARGET_COL]
      .rank(method="average", pct=True)
      - 0.5
)

# 每日横截面 z-score target，可作为备用
df["target_cs_zscore"] = (
    df.groupby("date")[TARGET_COL]
      .transform(lambda x: (x - x.mean()) / (x.std(ddof=0) + 1e-12))
)

# 默认使用 rank target
if MODEL_TARGET_MODE == "cs_rank":
    MODEL_TARGET_COL = "target_cs_rank"
elif MODEL_TARGET_MODE == "raw_return":
    MODEL_TARGET_COL = TARGET_COL
else:
    raise ValueError("MODEL_TARGET_MODE 只能是 'cs_rank' 或 'raw_return'。")

df = df[df[MODEL_TARGET_COL].notna()].copy()

print("\nOriginal return target:", TARGET_COL)
print("Model training target:", MODEL_TARGET_COL)
print("Model target mode:", MODEL_TARGET_MODE)
print("Data shape after target cleaning:", df.shape)


# ============================================================
# 5. 读取 75 个特征
# ============================================================

def load_feature_list(feature_file, data):
    """
    优先读取 boruta75_feature_cols.txt。
    如果文件不存在，则从 parquet 中自动推断数值型特征。
    """
    if feature_file.exists():
        print("\nUsing Boruta 75 feature list from:")
        print(feature_file)

        raw_lines = feature_file.read_text(encoding="utf-8").splitlines()

        features = []
        for line in raw_lines:
            line = line.strip()
            line = line.replace(",", "")
            line = line.replace('"', "")
            line = line.replace("'", "")

            if line == "":
                continue
            if line.lower() in ["feature", "features", "column", "columns"]:
                continue

            features.append(line)

        return features

    print("\nBoruta feature list file not found.")
    print("Automatically inferring numeric feature columns from the dataset.")

    exclude_cols = {
        "date",
        "ticker",
        "instrument_id",
        "sample_split",
        "is_trade_eligible",
        "target_r_on_next",
        "target_r_on_next_winsor",
        "target_cs_rank",
        "target_cs_zscore",
    }

    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()

    features = []
    for col in numeric_cols:
        col_lower = col.lower()

        if col in exclude_cols:
            continue
        if "target" in col_lower:
            continue
        if "future" in col_lower:
            continue
        if "next" in col_lower:
            continue
        if "lead" in col_lower:
            continue
        if "pred" in col_lower:
            continue
        if "score" in col_lower:
            continue

        features.append(col)

    return features


feature_cols = load_feature_list(FEATURE_FILE, df)

missing_features = [c for c in feature_cols if c not in df.columns]

if len(missing_features) > 0:
    print("\nWarning: these features are in the feature list but not in the dataset:")
    for c in missing_features[:50]:
        print(" -", c)

    feature_cols = [c for c in feature_cols if c in df.columns]

non_numeric_features = [
    c for c in feature_cols
    if not pd.api.types.is_numeric_dtype(df[c])
]

if len(non_numeric_features) > 0:
    print("\nWarning: removing non-numeric features:")
    for c in non_numeric_features:
        print(" -", c)

    feature_cols = [c for c in feature_cols if c not in non_numeric_features]

print("\nFinal number of XGBoost2 features:", len(feature_cols))

if len(feature_cols) == 0:
    raise ValueError("没有可用特征，请检查 feature list 或 parquet 数据。")

feature_list_df = pd.DataFrame({"feature": feature_cols})
feature_list_df.to_csv(OUTPUT_DIR / "xgboost2_feature_list_used.csv", index=False)


# ============================================================
# 6. 划分 train / validation / test
# ============================================================

def build_train_valid_test_masks(data):
    """
    优先使用 sample_split。
    如果没有 sample_split，则按日期做 60% / 20% / 20% 时间切分。
    """
    if "sample_split" in data.columns:
        split = data["sample_split"].astype(str).str.lower()

        train_mask = split.isin(["train", "training"])
        valid_mask = split.isin(["valid", "validation", "val"])
        test_mask = split.isin(["test", "oos", "out_of_sample"])

        if train_mask.sum() > 0 and valid_mask.sum() > 0 and test_mask.sum() > 0:
            print("\nUsing existing sample_split column.")
            return train_mask, valid_mask, test_mask

        print("\nsample_split exists but cannot identify train / validation / test.")
        print("Creating time-based split instead.")

    unique_dates = np.array(sorted(data["date"].unique()))
    n_dates = len(unique_dates)

    train_end = int(n_dates * 0.60)
    valid_end = int(n_dates * 0.80)

    train_dates = unique_dates[:train_end]
    valid_dates = unique_dates[train_end:valid_end]
    test_dates = unique_dates[valid_end:]

    train_mask = data["date"].isin(train_dates)
    valid_mask = data["date"].isin(valid_dates)
    test_mask = data["date"].isin(test_dates)

    data["sample_split"] = "unused"
    data.loc[train_mask, "sample_split"] = "train"
    data.loc[valid_mask, "sample_split"] = "validation"
    data.loc[test_mask, "sample_split"] = "test"

    print("\nCreated time-based split by date.")

    return train_mask, valid_mask, test_mask


train_mask, valid_mask, test_mask = build_train_valid_test_masks(df)

train_index = df.index[train_mask].to_numpy()
valid_index = df.index[valid_mask].to_numpy()
test_index = df.index[test_mask].to_numpy()

print("\nSplit summary:")
print("Train rows:", len(train_index))
print("Validation rows:", len(valid_index))
print("Test rows:", len(test_index))

print("\nTrain date range:", df.loc[train_mask, "date"].min(), "to", df.loc[train_mask, "date"].max())
print("Valid date range:", df.loc[valid_mask, "date"].min(), "to", df.loc[valid_mask, "date"].max())
print("Test date range:", df.loc[test_mask, "date"].min(), "to", df.loc[test_mask, "date"].max())


# ============================================================
# 7. 构造调参样本
# ============================================================

rng = np.random.default_rng(RANDOM_STATE)

if len(train_index) > TUNING_MAX_TRAIN_ROWS:
    tuning_train_index = rng.choice(
        train_index,
        size=TUNING_MAX_TRAIN_ROWS,
        replace=False,
    )
    tuning_train_index = np.sort(tuning_train_index)
else:
    tuning_train_index = train_index

if TUNING_VALID_FRAC < 1.0:
    valid_sample_size = int(len(valid_index) * TUNING_VALID_FRAC)
    tuning_valid_index = rng.choice(
        valid_index,
        size=valid_sample_size,
        replace=False,
    )
    tuning_valid_index = np.sort(tuning_valid_index)
else:
    tuning_valid_index = valid_index

print("\nTuning train rows:", len(tuning_train_index))
print("Tuning validation rows:", len(tuning_valid_index))

X_train_tune = df.loc[tuning_train_index, feature_cols].astype("float32")
y_train_tune = df.loc[tuning_train_index, MODEL_TARGET_COL].astype("float32")

X_valid_tune = df.loc[tuning_valid_index, feature_cols].astype("float32")
y_valid_tune = df.loc[tuning_valid_index, MODEL_TARGET_COL].astype("float32")

valid_tune_eval = df.loc[
    tuning_valid_index,
    ["date", TARGET_COL, MODEL_TARGET_COL]
].copy()

if REALIZED_RETURN_COL_CANDIDATE in df.columns:
    REALIZED_RETURN_COL = REALIZED_RETURN_COL_CANDIDATE
else:
    REALIZED_RETURN_COL = TARGET_COL

valid_tune_eval[REALIZED_RETURN_COL] = df.loc[tuning_valid_index, REALIZED_RETURN_COL]

print("\nX_train_tune shape:", X_train_tune.shape)
print("X_valid_tune shape:", X_valid_tune.shape)
print("Realized return column for long-short:", REALIZED_RETURN_COL)


# ============================================================
# 8. 评价函数
# ============================================================

def regression_metrics(y_true, y_pred):
    """
    常规回归指标。
    注意：
    如果 MODEL_TARGET_MODE = 'cs_rank'，
    这里评价的是 rank target 的拟合，不是收益率本身。
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return {
            "RMSE": np.nan,
            "MAE": np.nan,
            "R2": np.nan,
            "Pearson_IC": np.nan,
            "Spearman_IC": np.nan,
        }

    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)

    try:
        r2 = r2_score(y_true, y_pred)
    except Exception:
        r2 = np.nan

    pearson_ic = pd.Series(y_true).corr(pd.Series(y_pred), method="pearson")
    spearman_ic = pd.Series(y_true).corr(pd.Series(y_pred), method="spearman")

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Pearson_IC": pearson_ic,
        "Spearman_IC": spearman_ic,
    }


def daily_ic_table(data, y_col, pred_col):
    """
    每天横截面计算 Pearson IC 和 Spearman IC。
    """
    rows = []

    for date, g in data.groupby("date"):
        g = g[[y_col, pred_col]].dropna()

        if len(g) < 5:
            continue

        pearson_ic = g[y_col].corr(g[pred_col], method="pearson")
        spearman_ic = g[y_col].corr(g[pred_col], method="spearman")

        rows.append({
            "date": date,
            "n_stocks": len(g),
            "daily_pearson_ic": pearson_ic,
            "daily_spearman_ic": spearman_ic,
        })

    return pd.DataFrame(rows)


def daily_ic_summary_from_table(daily_ic):
    """
    对 daily IC 做汇总。
    """
    if len(daily_ic) == 0:
        return {
            "mean_daily_pearson_ic": np.nan,
            "mean_daily_spearman_ic": np.nan,
            "std_daily_spearman_ic": np.nan,
            "n_days": 0,
            "ic_ir": np.nan,
        }

    mean_p = daily_ic["daily_pearson_ic"].mean()
    mean_s = daily_ic["daily_spearman_ic"].mean()
    std_s = daily_ic["daily_spearman_ic"].std()
    n_days = daily_ic["date"].nunique()

    ic_ir = mean_s / std_s if std_s != 0 else np.nan

    return {
        "mean_daily_pearson_ic": mean_p,
        "mean_daily_spearman_ic": mean_s,
        "std_daily_spearman_ic": std_s,
        "n_days": n_days,
        "ic_ir": ic_ir,
    }


def long_short_backtest(data, realized_return_col, pred_col, q=0.10):
    """
    简单 long-short 检验：
    每天按照预测分数排序，
    做多 top 10%，做空 bottom 10%。
    """
    rows = []

    for date, g in data.groupby("date"):
        g = g[[realized_return_col, pred_col]].dropna().copy()

        if len(g) < 20:
            continue

        k = max(int(len(g) * q), 1)

        g = g.sort_values(pred_col)

        short_return = g.head(k)[realized_return_col].mean()
        long_return = g.tail(k)[realized_return_col].mean()

        long_short_return = long_return - short_return

        rows.append({
            "date": date,
            "n_stocks": len(g),
            "long_leg_return": long_return,
            "short_leg_return": short_return,
            "long_short_return": long_short_return,
        })

    out = pd.DataFrame(rows)

    if len(out) > 0:
        out["cum_long_short_return"] = (1 + out["long_short_return"]).cumprod() - 1

    return out


def long_short_summary_from_table(ls_table):
    """
    long-short 日收益汇总。
    """
    if len(ls_table) == 0:
        return {
            "mean_daily_long_short_return": np.nan,
            "std_daily_long_short_return": np.nan,
            "annualized_return_approx": np.nan,
            "annualized_vol_approx": np.nan,
            "sharpe_approx": np.nan,
            "positive_day_ratio": np.nan,
            "n_days": 0,
        }

    mean_ret = ls_table["long_short_return"].mean()
    std_ret = ls_table["long_short_return"].std()
    ann_ret = mean_ret * 252
    ann_vol = std_ret * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol != 0 else np.nan
    pos_ratio = (ls_table["long_short_return"] > 0).mean()

    return {
        "mean_daily_long_short_return": mean_ret,
        "std_daily_long_short_return": std_ret,
        "annualized_return_approx": ann_ret,
        "annualized_vol_approx": ann_vol,
        "sharpe_approx": sharpe,
        "positive_day_ratio": pos_ratio,
        "n_days": ls_table["date"].nunique(),
    }


def prediction_distribution_by_date(data, pred_col):
    """
    检查每天预测值的区分度。
    这个用于避免上一版出现每天预测值只有很少几个不同取值的问题。
    """
    rows = []

    for date, g in data.groupby("date"):
        pred = g[pred_col].dropna()

        if len(pred) == 0:
            continue

        rows.append({
            "date": date,
            "n_obs": len(pred),
            "n_unique_pred": pred.nunique(),
            "pred_mean": pred.mean(),
            "pred_std": pred.std(),
            "pred_min": pred.min(),
            "pred_p01": pred.quantile(0.01),
            "pred_p10": pred.quantile(0.10),
            "pred_p50": pred.quantile(0.50),
            "pred_p90": pred.quantile(0.90),
            "pred_p99": pred.quantile(0.99),
            "pred_max": pred.max(),
        })

    return pd.DataFrame(rows)


# ============================================================
# 9. 更丰富的 XGBoost 参数组合
# ============================================================

param_grid = [
    # 较浅树，较低正则化
    {"n_estimators": 100, "max_depth": 2, "learning_rate": 0.05, "subsample": 0.90, "colsample_bytree": 0.90, "min_child_weight": 1,  "reg_lambda": 1,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 200, "max_depth": 2, "learning_rate": 0.03, "subsample": 0.90, "colsample_bytree": 0.90, "min_child_weight": 1,  "reg_lambda": 1,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 400, "max_depth": 2, "learning_rate": 0.02, "subsample": 0.90, "colsample_bytree": 0.90, "min_child_weight": 3,  "reg_lambda": 3,  "reg_alpha": 0,   "gamma": 0},

    # 中等深度，增强学习能力
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.05, "subsample": 0.90, "colsample_bytree": 0.90, "min_child_weight": 1,  "reg_lambda": 1,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.03, "subsample": 0.90, "colsample_bytree": 0.90, "min_child_weight": 1,  "reg_lambda": 1,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 400, "max_depth": 3, "learning_rate": 0.02, "subsample": 0.90, "colsample_bytree": 0.90, "min_child_weight": 3,  "reg_lambda": 3,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 600, "max_depth": 3, "learning_rate": 0.01, "subsample": 0.90, "colsample_bytree": 0.90, "min_child_weight": 5,  "reg_lambda": 5,  "reg_alpha": 0,   "gamma": 0},

    # 中等深度，稍强正则化
    {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.03, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 5,  "reg_lambda": 5,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 400, "max_depth": 3, "learning_rate": 0.02, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 5,  "reg_lambda": 10, "reg_alpha": 0.1, "gamma": 0},
    {"n_estimators": 600, "max_depth": 3, "learning_rate": 0.01, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 10, "reg_lambda": 10, "reg_alpha": 0.1, "gamma": 0},

    # 更深树，允许更多非线性
    {"n_estimators": 100, "max_depth": 4, "learning_rate": 0.05, "subsample": 0.90, "colsample_bytree": 0.90, "min_child_weight": 1,  "reg_lambda": 1,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.03, "subsample": 0.90, "colsample_bytree": 0.90, "min_child_weight": 3,  "reg_lambda": 3,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 400, "max_depth": 4, "learning_rate": 0.02, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 5,  "reg_lambda": 5,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 600, "max_depth": 4, "learning_rate": 0.01, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 5,  "reg_lambda": 10, "reg_alpha": 0.1, "gamma": 0},

    # 更强正则化的深树，用于防止过拟合
    {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.03, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 10, "reg_lambda": 10, "reg_alpha": 0.1, "gamma": 0.1},
    {"n_estimators": 400, "max_depth": 4, "learning_rate": 0.02, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 10, "reg_lambda": 20, "reg_alpha": 0.1, "gamma": 0.1},
    {"n_estimators": 600, "max_depth": 4, "learning_rate": 0.01, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 20, "reg_lambda": 20, "reg_alpha": 1.0, "gamma": 0.1},

    # 深度 5，测试更强表达能力
    {"n_estimators": 100, "max_depth": 5, "learning_rate": 0.05, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 3,  "reg_lambda": 3,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.03, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 5,  "reg_lambda": 5,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 400, "max_depth": 5, "learning_rate": 0.02, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 10, "reg_lambda": 10, "reg_alpha": 0.1, "gamma": 0.1},

    # 更慢学习率 + 更多树
    {"n_estimators": 800, "max_depth": 2, "learning_rate": 0.01, "subsample": 0.90, "colsample_bytree": 0.90, "min_child_weight": 3,  "reg_lambda": 3,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 800, "max_depth": 3, "learning_rate": 0.01, "subsample": 0.85, "colsample_bytree": 0.85, "min_child_weight": 5,  "reg_lambda": 5,  "reg_alpha": 0,   "gamma": 0},
    {"n_estimators": 800, "max_depth": 4, "learning_rate": 0.01, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 10, "reg_lambda": 10, "reg_alpha": 0.1, "gamma": 0.1},
]

print("\nNumber of XGBoost2 parameter combinations:", len(param_grid))


# ============================================================
# 10. 训练和调参
# ============================================================

def fit_xgb_model(params):
    """
    训练 XGBoost。
    本版不使用 early stopping，避免模型在第 0 轮过早停止。
    """
    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        tree_method="hist",
        max_bin=256,
        n_jobs=-1,
        eval_metric="rmse",
        importance_type="gain",
        missing=np.nan,
        **params,
    )

    model.fit(
        X_train_tune,
        y_train_tune,
        verbose=False,
    )

    return model


tuning_rows = []
models = []

for i, params in enumerate(param_grid, start=1):
    print("\n" + "=" * 100)
    print(f"Training XGBoost2 candidate model {i}/{len(param_grid)}")
    print(params)

    model = fit_xgb_model(params)

    valid_pred = model.predict(X_valid_tune)

    # 1. 对模型训练目标本身的回归指标
    model_target_metrics = regression_metrics(y_valid_tune, valid_pred)

    # 2. 对真实收益率的 pooled IC
    pooled_return_pearson_ic = pd.Series(valid_tune_eval[TARGET_COL].values).corr(
        pd.Series(valid_pred),
        method="pearson",
    )

    pooled_return_spearman_ic = pd.Series(valid_tune_eval[TARGET_COL].values).corr(
        pd.Series(valid_pred),
        method="spearman",
    )

    # 3. daily cross-sectional IC
    eval_temp = valid_tune_eval.copy()
    eval_temp["pred_xgboost2"] = valid_pred

    daily_ic = daily_ic_table(
        data=eval_temp,
        y_col=TARGET_COL,
        pred_col="pred_xgboost2",
    )

    daily_ic_summary = daily_ic_summary_from_table(daily_ic)

    # 4. simple long-short
    ls_table = long_short_backtest(
        data=eval_temp,
        realized_return_col=REALIZED_RETURN_COL,
        pred_col="pred_xgboost2",
        q=0.10,
    )

    ls_summary = long_short_summary_from_table(ls_table)

    # 5. 预测值区分度
    pred_dist = prediction_distribution_by_date(
        data=eval_temp,
        pred_col="pred_xgboost2",
    )

    if len(pred_dist) > 0:
        mean_unique_pred_per_day = pred_dist["n_unique_pred"].mean()
        median_unique_pred_per_day = pred_dist["n_unique_pred"].median()
        mean_pred_std_per_day = pred_dist["pred_std"].mean()
    else:
        mean_unique_pred_per_day = np.nan
        median_unique_pred_per_day = np.nan
        mean_pred_std_per_day = np.nan

    row = {
        "model_id": i,
        **params,

        "model_target_RMSE": model_target_metrics["RMSE"],
        "model_target_MAE": model_target_metrics["MAE"],
        "model_target_R2": model_target_metrics["R2"],
        "model_target_Pearson_IC": model_target_metrics["Pearson_IC"],
        "model_target_Spearman_IC": model_target_metrics["Spearman_IC"],

        "valid_pooled_return_pearson_ic": pooled_return_pearson_ic,
        "valid_pooled_return_spearman_ic": pooled_return_spearman_ic,

        "valid_mean_daily_pearson_ic": daily_ic_summary["mean_daily_pearson_ic"],
        "valid_mean_daily_spearman_ic": daily_ic_summary["mean_daily_spearman_ic"],
        "valid_std_daily_spearman_ic": daily_ic_summary["std_daily_spearman_ic"],
        "valid_ic_ir": daily_ic_summary["ic_ir"],
        "valid_n_ic_days": daily_ic_summary["n_days"],

        "valid_mean_daily_long_short_return": ls_summary["mean_daily_long_short_return"],
        "valid_long_short_sharpe": ls_summary["sharpe_approx"],
        "valid_long_short_positive_day_ratio": ls_summary["positive_day_ratio"],

        "valid_mean_unique_pred_per_day": mean_unique_pred_per_day,
        "valid_median_unique_pred_per_day": median_unique_pred_per_day,
        "valid_mean_pred_std_per_day": mean_pred_std_per_day,
    }

    tuning_rows.append(row)
    models.append(model)

    print("Validation mean daily Spearman IC:", row["valid_mean_daily_spearman_ic"])
    print("Validation ICIR:", row["valid_ic_ir"])
    print("Validation long-short Sharpe:", row["valid_long_short_sharpe"])
    print("Mean unique predictions per day:", row["valid_mean_unique_pred_per_day"])


tuning_df = pd.DataFrame(tuning_rows)

# 最重要的选择标准：
# 1. validation mean daily Spearman IC
# 2. validation long-short Sharpe
# 3. validation ICIR
# 这比上一版用 pooled Spearman IC 更符合每日横截面排序任务。
tuning_df = tuning_df.sort_values(
    by=[
        "valid_mean_daily_spearman_ic",
        "valid_long_short_sharpe",
        "valid_ic_ir",
    ],
    ascending=[False, False, False],
).reset_index(drop=True)

best_model_id = int(tuning_df.loc[0, "model_id"])
best_params = param_grid[best_model_id - 1]

print("\n" + "=" * 100)
print("Best XGBoost2 model selected by validation mean daily Spearman IC")
print("Best model ID:", best_model_id)
print("Best parameters:")
print(best_params)
print("\nBest validation row:")
print(tuning_df.head(1))

tuning_df.to_csv(OUTPUT_DIR / "xgboost2_tuning_results.csv", index=False)


# ============================================================
# 11. 用最优参数在完整 train set 上重新训练最终模型
# ============================================================

if REFIT_FINAL_ON_TRAIN_VALID:
    final_train_index = np.sort(np.concatenate([train_index, valid_index]))
    final_train_description = "train_plus_validation"
else:
    final_train_index = train_index
    final_train_description = "train_only"

print("\n" + "=" * 100)
print("Refitting final XGBoost2 model")
print("Final training sample:", final_train_description)
print("Final training rows:", len(final_train_index))
print("Best params:", best_params)

X_train_final = df.loc[final_train_index, feature_cols].astype("float32")
y_train_final = df.loc[final_train_index, MODEL_TARGET_COL].astype("float32")

final_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    tree_method="hist",
    max_bin=256,
    n_jobs=-1,
    eval_metric="rmse",
    importance_type="gain",
    missing=np.nan,
    **best_params,
)

final_model.fit(
    X_train_final,
    y_train_final,
    verbose=False,
)

print("Final model training finished.")


# ============================================================
# 12. 生成 train / validation / test 预测
# ============================================================

PRED_COL = "pred_xgboost2"
RANK_COL = "score_xgboost2_rank"

df[PRED_COL] = np.nan

for split_name, idx in [
    ("train", train_index),
    ("validation", valid_index),
    ("test", test_index),
]:
    print(f"Predicting {split_name} set...")

    X_temp = df.loc[idx, feature_cols].astype("float32")
    df.loc[idx, PRED_COL] = final_model.predict(X_temp)

# 每天横截面 rank score
# 越接近 1，说明模型越看好
# 越接近 0，说明模型越不看好
df[RANK_COL] = df.groupby("date")[PRED_COL].rank(
    method="average",
    pct=True,
)

print("\nPrediction finished.")
print(df[["date", TARGET_COL, MODEL_TARGET_COL, PRED_COL, RANK_COL]].head())


# ============================================================
# 13. 最终整体指标
# ============================================================

metrics_rows = []

for split_name, mask in [
    ("train", train_mask),
    ("validation", valid_mask),
    ("test", test_mask),
]:
    temp = df.loc[mask].copy()

    # 对模型训练目标的指标
    model_metrics = regression_metrics(
        temp[MODEL_TARGET_COL],
        temp[PRED_COL],
    )

    # 对真实收益率的 pooled IC
    pooled_return_pearson_ic = temp[TARGET_COL].corr(temp[PRED_COL], method="pearson")
    pooled_return_spearman_ic = temp[TARGET_COL].corr(temp[PRED_COL], method="spearman")

    row = {
        "split": split_name,
        "model_target": MODEL_TARGET_COL,

        "model_target_RMSE": model_metrics["RMSE"],
        "model_target_MAE": model_metrics["MAE"],
        "model_target_R2": model_metrics["R2"],
        "model_target_Pearson_IC": model_metrics["Pearson_IC"],
        "model_target_Spearman_IC": model_metrics["Spearman_IC"],

        "pooled_return_Pearson_IC": pooled_return_pearson_ic,
        "pooled_return_Spearman_IC": pooled_return_spearman_ic,
    }

    metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows)

print("\n" + "=" * 100)
print("Overall metrics:")
print(metrics_df)

metrics_df.to_csv(OUTPUT_DIR / "xgboost2_metrics.csv", index=False)


# ============================================================
# 14. 最终 daily IC
# ============================================================

daily_ic_all = []

for split_name, mask in [
    ("train", train_mask),
    ("validation", valid_mask),
    ("test", test_mask),
]:
    temp = df.loc[mask].copy()

    daily_ic = daily_ic_table(
        data=temp,
        y_col=TARGET_COL,
        pred_col=PRED_COL,
    )

    daily_ic["split"] = split_name
    daily_ic_all.append(daily_ic)

daily_ic_df = pd.concat(daily_ic_all, ignore_index=True)

summary_rows = []

for split_name, g in daily_ic_df.groupby("split"):
    s = daily_ic_summary_from_table(g)
    s["split"] = split_name
    summary_rows.append(s)

daily_ic_summary = pd.DataFrame(summary_rows)
daily_ic_summary = daily_ic_summary[
    [
        "split",
        "mean_daily_pearson_ic",
        "mean_daily_spearman_ic",
        "std_daily_spearman_ic",
        "ic_ir",
        "n_days",
    ]
]

print("\n" + "=" * 100)
print("Daily IC summary:")
print(daily_ic_summary)

daily_ic_df.to_csv(OUTPUT_DIR / "xgboost2_daily_ic.csv", index=False)
daily_ic_summary.to_csv(OUTPUT_DIR / "xgboost2_daily_ic_summary.csv", index=False)


# ============================================================
# 15. 最终 long-short 检验
# ============================================================

long_short_all = []

for split_name, mask in [
    ("train", train_mask),
    ("validation", valid_mask),
    ("test", test_mask),
]:
    temp = df.loc[mask].copy()

    ls = long_short_backtest(
        data=temp,
        realized_return_col=REALIZED_RETURN_COL,
        pred_col=PRED_COL,
        q=0.10,
    )

    ls["split"] = split_name
    long_short_all.append(ls)

long_short_df = pd.concat(long_short_all, ignore_index=True)

ls_summary_rows = []

for split_name, g in long_short_df.groupby("split"):
    s = long_short_summary_from_table(g)
    s["split"] = split_name
    ls_summary_rows.append(s)

long_short_summary = pd.DataFrame(ls_summary_rows)

long_short_summary = long_short_summary[
    [
        "split",
        "mean_daily_long_short_return",
        "std_daily_long_short_return",
        "annualized_return_approx",
        "annualized_vol_approx",
        "sharpe_approx",
        "positive_day_ratio",
        "n_days",
    ]
]

print("\n" + "=" * 100)
print("Long-short summary:")
print(long_short_summary)

long_short_df.to_csv(OUTPUT_DIR / "xgboost2_long_short_daily_returns.csv", index=False)
long_short_summary.to_csv(OUTPUT_DIR / "xgboost2_long_short_summary.csv", index=False)


# ============================================================
# 16. 预测分数区分度检查
# ============================================================

pred_dist_all = []

for split_name, mask in [
    ("train", train_mask),
    ("validation", valid_mask),
    ("test", test_mask),
]:
    temp = df.loc[mask].copy()

    pred_dist = prediction_distribution_by_date(
        data=temp,
        pred_col=PRED_COL,
    )

    pred_dist["split"] = split_name
    pred_dist_all.append(pred_dist)

prediction_distribution_df = pd.concat(pred_dist_all, ignore_index=True)

prediction_distribution_summary = (
    prediction_distribution_df
    .groupby("split")
    .agg(
        mean_unique_pred_per_day=("n_unique_pred", "mean"),
        median_unique_pred_per_day=("n_unique_pred", "median"),
        mean_pred_std_per_day=("pred_std", "mean"),
        median_pred_std_per_day=("pred_std", "median"),
        n_days=("date", "nunique"),
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("Prediction distribution summary:")
print(prediction_distribution_summary)

prediction_distribution_df.to_csv(
    OUTPUT_DIR / "xgboost2_prediction_distribution_by_date.csv",
    index=False,
)

prediction_distribution_summary.to_csv(
    OUTPUT_DIR / "xgboost2_prediction_distribution_summary.csv",
    index=False,
)


# ============================================================
# 17. 特征重要性
# ============================================================

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_gain_normalized": final_model.feature_importances_,
})

# booster gain
booster_score = final_model.get_booster().get_score(importance_type="gain")

importance_df["importance_gain_raw"] = importance_df["feature"].map(booster_score).fillna(0.0)

importance_df = importance_df.sort_values(
    "importance_gain_raw",
    ascending=False,
).reset_index(drop=True)

print("\n" + "=" * 100)
print("Top 30 XGBoost2 feature importance:")
print(importance_df.head(30))

importance_df.to_csv(
    OUTPUT_DIR / "xgboost2_feature_importance_gain.csv",
    index=False,
)


# ============================================================
# 18. 保存预测结果
# ============================================================

keep_cols = []

for col in [
    "date",
    "ticker",
    "instrument_id",
    "sample_split",
    "is_trade_eligible",
    "target_r_on_next",
    "target_r_on_next_winsor",
    "target_cs_rank",
    "target_cs_zscore",
    PRED_COL,
    RANK_COL,
]:
    if col in df.columns:
        keep_cols.append(col)

predictions_df = df[keep_cols].copy()

# 保存全部样本预测
predictions_df.to_parquet(
    OUTPUT_DIR / "xgboost2_predictions_all.parquet",
    index=False,
)

# 保存 test 样本预测
test_predictions_df = predictions_df.loc[test_mask].copy()

test_predictions_df.to_parquet(
    OUTPUT_DIR / "xgboost2_predictions_test.parquet",
    index=False,
)

test_predictions_df.to_csv(
    OUTPUT_DIR / "xgboost2_predictions_test.csv",
    index=False,
)

print("\n" + "=" * 100)
print("Test prediction preview:")
print(test_predictions_df.head())


# ============================================================
# 19. 保存模型和配置
# ============================================================

with open(OUTPUT_DIR / "xgboost2_best_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

run_config = {
    "target_col": TARGET_COL,
    "realized_return_col": REALIZED_RETURN_COL,
    "model_target_mode": MODEL_TARGET_MODE,
    "model_target_col": MODEL_TARGET_COL,
    "random_state": RANDOM_STATE,
    "tuning_max_train_rows": TUNING_MAX_TRAIN_ROWS,
    "tuning_valid_frac": TUNING_VALID_FRAC,
    "refit_final_on_train_valid": REFIT_FINAL_ON_TRAIN_VALID,
    "final_train_description": final_train_description,
    "best_model_id": best_model_id,
    "best_params": best_params,
    "n_features": len(feature_cols),
    "n_param_combinations": len(param_grid),
    "input_data_path": str(DATA_PATH),
    "output_dir": str(OUTPUT_DIR),
}

with open(OUTPUT_DIR / "xgboost2_run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=4)

print("\n" + "=" * 100)
print("XGBoost2 training finished.")
print("All output files are saved in:")
print(OUTPUT_DIR)

print("\nGenerated files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print(" -", file.name)

print("=" * 100)

Project directory:
C:\Users\yuchao\Desktop\MLF_coursework

Input data path:
C:\Users\yuchao\Desktop\MLF_coursework\stage4_final_75_features_panel.parquet

Feature list path:
C:\Users\yuchao\Desktop\MLF_coursework\stage4_feature_selection_outputs\boruta75_feature_cols.txt

Output directory:
C:\Users\yuchao\Desktop\MLF_coursework\stage4_feature_selection_outputs\XGBoost2_prediction_outcome
Original data shape: (3773971, 83)
Date range: 2010-01-04 00:00:00 to 2024-12-31 00:00:00
Number of columns: 83

Filtered by is_trade_eligible == True
Rows before: 3773971
Rows after: 3171067

Original return target: target_r_on_next_winsor
Model training target: target_cs_rank
Model target mode: cs_rank
Data shape after target cleaning: (3170073, 85)

Using Boruta 75 feature list from:
C:\Users\yuchao\Desktop\MLF_coursework\stage4_feature_selection_outputs\boruta75_feature_cols.txt

Final number of XGBoost2 features: 75

Using existing sample_split column.

Split summary:
Train rows: 1758771
Validatio

训练后模型选择的最优参数组合是第20组：

| 参数                 |   取值 |
| ------------------ | ---: |
| `n_estimators`     |  400 |
| `max_depth`        |    5 |
| `learning_rate`    | 0.02 |
| `subsample`        | 0.80 |
| `colsample_bytree` | 0.80 |
| `min_child_weight` |   10 |
| `reg_lambda`       |   10 |
| `reg_alpha`        |  0.1 |
| `gamma`            |  0.1 |

这组参数在验证集上表现为：

valid_mean_daily_spearman_ic = 0.032224

valid_ic_ir = 0.158631

valid_long_short_sharpe = 1.142490

代码输出的结果里对portfolio construction最有用的是xgboost2_predictions_test.csv的以下几列：

date

ticker 是股票代码

instrument_id 股票的唯一 ID

pred_xgboost2模型输出的原始预测分数（XGBoost2 学的是这只股票在当天横截面里未来收益排名会更靠前还是更靠后。因此分数越高就代表这支股票的收益排名越靠前）

score_xgboost2_rank XGBoost2on 使用的列。它是代码把 pred_xgboost2 在每天横截面内重新做排名后得到的分数，它的取值范围大致是0 到 1
| `score_xgboost2_rank` | 含义          |
| --------------------: | ----------- |
|                  接近 1 | 当天模型最看好的股票  |
|                接近 0.5 | 当天排名中间的股票   |
|                  接近 0 | 当天模型最不看好的股票 |


这几列的含义综合起来就是：在某一天 date，对某只股票 ticker / instrument_id，XGBoost2 给出了一个预测分数 pred_xgboost2，再把这个分数转换成当天横截面排名 score_xgboost2_rank。后续组合构建就可以根据这个排名决定做多谁、做空谁。